# D-FINE-N reproduction


In [ ]:
import subprocess

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(gpu)
if not gpu.startswith("Tesla T4,"):
    raise RuntimeError("select a Tesla T4 runtime for this reproduction")

In [ ]:
import os
from pathlib import Path

installer = Path("/tmp/uv-install.sh")
subprocess.run(
    [
        "curl",
        "-LsSf",
        "https://astral.sh/uv/0.12.23/install.sh",
        "-o",
        str(installer),
    ],
    check=True,
)
env = os.environ.copy()
env["UV_UNMANAGED_INSTALL"] = "/usr/local/bin"
subprocess.run(["sh", str(installer)], check=True, env=env)

uv_version = subprocess.run(
    ["uv", "--version"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(uv_version)
if not uv_version.startswith("uv 0.12.23 "):
    raise RuntimeError(f"unexpected uv version: {uv_version}")

In [ ]:
import shutil

repository = "https://github.com/aekn/smallpotato.git"
revision = "071a4a66acd9e8b3ae3b523110a6becfe208b304"
repo = Path("/content/smallpotato")

if repo.exists():
    shutil.rmtree(repo)

subprocess.run(["git", "init", "-q", str(repo)], check=True)
subprocess.run(
    ["git", "-C", str(repo), "remote", "add", "origin", repository],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(repo),
        "fetch",
        "-q",
        "--depth=1",
        "--no-tags",
        "origin",
        revision,
    ],
    check=True,
)
subprocess.run(
    ["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"],
    check=True,
)

actual = subprocess.run(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if actual != revision:
    raise RuntimeError(f"expected {revision}, got {actual}")

status = subprocess.run(
    ["git", "-C", str(repo), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout
if status:
    raise RuntimeError("repository checkout is dirty")

os.chdir(repo)
print(f"{repo} @ {actual}")

In [ ]:
subprocess.run(["uv", "sync", "--locked"], cwd=repo, check=True)
print("environment ready")

In [ ]:
process = subprocess.Popen(
    ["uv", "run", "python", "-m", "reproductions.dfine.reproduce"],
    cwd=repo,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, process.args)

In [ ]:
runs = repo / "runs" / "dfine"
run_dir = max(
    (path for path in runs.iterdir() if path.is_dir()),
    key=lambda path: path.name,
)
print(run_dir)

for name in ("summary.json", "provenance.json"):
    print(f"\n{name}")
    print((run_dir / name).read_text(encoding="utf-8"), end="")

In [ ]:
from google.colab import files

archive_base = Path("/content") / f"dfine-{run_dir.name}"
run_archive = shutil.make_archive(
    str(archive_base),
    "gztar",
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)
print(run_archive)
files.download(run_archive)